In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch01-smiles-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["rdkit==2026.9.1", "pillow==13.0.0", "numpy==1.26.4"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "01-02-smiles.py"

# 從紙上的鍵寫到 SMILES

先畫苯酚的兩種寫法，再畫只含環上單鍵的對照。預測哪兩串相同，再查看下格的原子、鍵與芳香性核對表。最後用乙醇及乳酸圖檢查走訪順序和立體配置。

In [ ]:
%%writefile "{script}"
# 概念示意。此處比較分子結構的寫法，沒有量測或建模資料。
from pathlib import Path
from rdkit.Chem import Draw

from rdkit import Chem

# 這兩張表用 RDKit 檢查寫法，不讀取量測資料。
print("SMILES | 元素 | 各原子所帶氫數 | 鍵型")
for text in ["O", "CCO", "Cl", "[NH4+]", "CC", "C-C", "C=C", "C#C"]:
    mol = Chem.MolFromSmiles(text)
    print(text, [a.GetSymbol() for a in mol.GetAtoms()],
          [a.GetTotalNumHs() for a in mol.GetAtoms()],
          [str(b.GetBondType()) for b in mol.GetBonds()])
print("SMILES | 標準寫法 | 芳香原子數")
for text in ["Oc1ccccc1", "OC1=CC=CC=C1", "OC1CCCCC1"]:
    mol = Chem.MolFromSmiles(text)
    print(text, Chem.MolToSmiles(mol), sum(a.GetIsAromatic() for a in mol.GetAtoms()))

for text in ["CCO", "OCC", "Oc1ccccc1"]:
    mol = Chem.MolFromSmiles(text)
    if mol is None:
        raise ValueError(f"無法讀取 {text}")
    print(text, "→", Chem.MolToSmiles(mol, canonical=True))

# 以固定順序畫圖，過程不使用隨機數。
texts = ["CCO", "OCC", "Oc1ccccc1",
         "C[C@H](O)C(=O)O", "C[C@@H](O)C(=O)O", "CC(O)C(=O)O"]
mols = [Chem.MolFromSmiles(text) for text in texts]
print("乳酸輸入、保留立體標記、略去立體標記")
for text, mol in zip(texts[3:], mols[3:]):
    print(text, Chem.MolToSmiles(mol), Chem.MolToSmiles(mol, isomericSmiles=False))
fig = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 210), legends=texts)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.save(out / "01-02-smiles.png")

In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "01-02-smiles.png")))